In [ ]:
# Imports & Config

import socket
import json
import time
import random
import ssl

HOST = '127.0.0.1'
PORT = 9999

In [ ]:
# Helpers

def send_msg(sock, msg: dict):
    data = json.dumps(msg) + '\n'
    sock.sendall(data.encode('utf-8'))


def recv_msg(sock) -> dict:
    buffer = ''
    while True:
        chunk = sock.recv(4096).decode('utf-8')
        if not chunk:
            raise ConnectionError('Server closed connection')
        buffer += chunk
        if '\n' in buffer:
            line, buffer = buffer.split('\n', 1)
            return json.loads(line)

In [ ]:
# Task Executor
def execute_task(payload: dict) -> str:
    task = payload.get('task')

    if task == 'sleep':
        duration = payload.get('duration', 1)
        print(f"Sleeping {duration}s")
        time.sleep(duration)
        return "done"

    elif task == 'math':
        a = payload.get('a', 0)
        b = payload.get('b', 0)
        op = payload.get('operation')
        if op == 'add':      return str(a + b)
        if op == 'subtract': return str(a - b)
        if op == 'multiply': return str(a * b)
        if op == 'divide':   return str(a / b) if b != 0 else 'division by zero'
        return 'unknown operation'

    elif task == 'fibonacci':
        n = payload.get('n', 10)
        a, b = 0, 1
        for _ in range(n):
            a, b = b, a + b
        return str(a)

    else:
        return "unknown task"

In [ ]:
# Failure Simulation
def maybe_crash(sock):
    if random.random() < 0.1:
        print("----- Worker crashed! -----")
        sock.close()
        raise ConnectionAbortedError("Simulated random crash")

In [ ]:
# Worker Main Loop
def start_worker():
    context = ssl.create_default_context()
    context.check_hostname = False
    context.verify_mode = ssl.CERT_NONE

    while True:
        try:
           
            raw_sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            sock = context.wrap_socket(raw_sock, server_hostname=HOST)
            
            sock.connect((HOST, PORT))

            send_msg(sock, {"type": "register"})
            response = recv_msg(sock)

            if response.get("type") != "registered":
                print("Registration failed")
                return

            worker_id = response.get("worker_id")
            print(f"\n--- Secure Worker Online | ID: {worker_id} ---")

            idle_logged = False

            while True:
                send_msg(sock, {"type": "request", "worker_id": worker_id})
                response = recv_msg(sock)

                if response.get("type") == "no_job":
                    if not idle_logged:
                        print("Idle - Waiting for queue...")
                        idle_logged = True
                    time.sleep(2)
                    continue
                
                idle_logged = False

                if response.get("type") == "assign":
                    job_id = response.get("job_id")
                    payload = response.get("payload")

                    maybe_crash(sock)

                    result = execute_task(payload)

                    send_msg(sock, {
                        "type": "complete",
                        "job_id": job_id,
                        "worker_id": worker_id,
                        "result": result
                    })

                    recv_msg(sock)
                    print(f"Encrypted Proc : {job_id} -> {result}")

        except Exception as e:
            print(f"[!] Worker Fault: {e} - Reconnecting in 3s...")
            time.sleep(3)

In [ ]:
# Run Worker
start_worker()